# Macro Factor Rotation - Research Notebook

**Source**: QC Strategy Library - AI Stocks-Bonds Rotation

**Concept**: Uses 3 FRED macro factors (VIX, yield curve, fed funds rate) to predict 21-day forward returns
for a multi-asset portfolio (SPY, GLD, BND, BTC). A DecisionTreeRegressor is trained per asset with
a 4-year lookback window. Positive predictions receive 1.5x leveraged weight.

In [ ]:
from AlgorithmImports import *
qb = QuantBook()

# Define universe
tickers = ["SPY", "GLD", "BND"]
for t in tickers:
    qb.AddEquity(t, Resolution.Daily)

print(f"Universe: {tickers} + BTCUSD")
print(f"Factors: VIXCLS (VIX), T10Y3M (Yield Curve), DFF (Fed Funds)")

## Strategy Logic

1. **Data**: FRED macro factors fetched daily (VIX, 10Y-3M spread, Fed Funds rate)
2. **Training**: Per-asset DecisionTreeRegressor (depth=12) on 4 years of history
3. **Prediction**: 21-day forward return forecast from latest factor values
4. **Allocation**: Assets with positive prediction get 1.5x leveraged weight
5. **BTC cap**: Bitcoin exposure capped at 10%, excess redistributed to equities
6. **Monthly rebalance**: Retrain and reallocate each month

In [ ]:
# Fetch historical data for analysis
spy_hist = qb.History("SPY", timedelta(252 * 10), Resolution.Daily)
gld_hist = qb.History("GLD", timedelta(252 * 10), Resolution.Daily)
bnd_hist = qb.History("BND", timedelta(252 * 10), Resolution.Daily)

print(f"SPY data: {spy_hist.shape[0]} days")
print(f"GLD data: {gld_hist.shape[0]} days")
print(f"BND data: {bnd_hist.shape[0]} days")

# Correlation analysis
returns = pd.DataFrame({
    "SPY": spy_hist["close"].pct_change(),
    "GLD": gld_hist["close"].pct_change(),
    "BND": bnd_hist["close"].pct_change(),
}).dropna()

print("\nCorrelation matrix (daily returns):")
print(returns.corr().round(2))

In [ ]:
# Analyze macro factor regimes
spy_close = spy_hist["close"]
spy_21d_fwd = spy_close.pct_change(21).shift(-21).dropna()

print("SPY 21-day forward return statistics:")
print(f"  Mean: {spy_21d_fwd.mean():.2%}")
print(f"  Std: {spy_21d_fwd.std():.2%}")
print(f"  Positive ratio: {(spy_21d_fwd > 0).mean():.1%}")
print(f"  Sharpe (annualized): {(spy_21d_fwd.mean() / spy_21d_fwd.std()) * (252/21)**0.5:.2f}")

## Key Observations

- **Macro factors** provide regime context: VIX for risk appetite, yield curve for recession signal, Fed funds for monetary stance
- **DecisionTreeRegressor** captures non-linear factor-return relationships
- **Per-asset models** allow independent prediction across asset classes
- **1.5x leverage** amplifies conviction but increases drawdown risk

### Strengths
- Interpretable model (decision tree) with macro economic intuition
- Multi-asset diversification (equity, gold, bonds, crypto)
- Monthly retraining adapts to changing macro regimes
- FRED data is free, reliable, and well-documented

### Risks
- DecisionTree can overfit with depth=12 on 4 years of data
- BTC cap at 10% limits crypto exposure but adds complexity
- 21-day forward labels introduce look-ahead bias risk
- Leverage (1.5x) amplifies losses during misprediction
- FRED data may have publication lag in live trading

In [ ]:
# Backtest results placeholder
print("=" * 50)
print("BACKTEST RESULTS (10-year)")
print("=" * 50)
print("Run backtest via QC MCP to populate metrics:")
print("  create_compile -> create_backtest -> read_backtest")
print()
print("Expected characteristics:")
print("  - CAGR: ~10-18% (leveraged multi-asset)")
print("  - Max Drawdown: ~15-25%")
print("  - Sharpe: ~0.8-1.2")
print("  - Rebalance frequency: Monthly")
print("  - Universe: 4 assets (SPY, GLD, BND, BTC)")
print("  - Factors: VIX, Yield Curve, Fed Funds")